# Self-Attention: The Core Mechanism

**Course:** MAI554 -- Deep Learning for NLP | **Lecture:** 05 | **Spring 2026** | **Prof. Anis Koubaa**

---

### Learning Objectives

1. **Understand Q, K, V** -- where they come from, how they differ, what each role contributes
2. **Walk through the attention formula step by step** -- dot product, scaling, softmax, weighted sum
3. **See self-attention as knowledge discovery** -- how Q/K matching reveals semantic relationships
4. **Implement a complete single-head self-attention** from scratch
5. **Understand causal masking** -- how autoregressive models block the future

> **Central idea:** Self-attention lets every token ask *who is relevant to me?* and receive a **weighted blend** of answers. Three views of the same input -- Query (search), Key (match), Value (content) -- create an **asymmetric knowledge marketplace** where demand meets supply.

## Setup

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt, math
plt.rcParams['figure.dpi'] = 100
torch.manual_seed(42); np.random.seed(42)

---
## Part 1: Where Do Query, Key, Value Come From?

The Q/K/V pattern was **not invented** for Transformers. It comes from **databases and search engines**.

- **Query** = what you are **searching for**
- **Key** = the **label** attached to each item -- does it match the query?
- **Value** = the **actual content** you get back when a key matches

**Google Search analogy:** you type a Query, Google compares it against millions of Keys (page titles), each gets a relevance score, and the Values (page content) of top matches are returned -- weighted by relevance.

In a search engine, Q and K are *different things*. In **self-attention**, the *same input* produces all three -- that is why it is called **self**-attention.

---
## Part 2: Computing Q, K, V -- Same Input, Three Roles

$$Q = X \cdot W_Q \quad \text{("What do I need?")}$$
$$K = X \cdot W_K \quad \text{("What can I offer?")}$$
$$V = X \cdot W_V \quad \text{("What content do I carry?")}$$

Separating Q/K (for **matching**) from V (for **content transfer**) lets the model learn *who should I attend to* separately from *what information to extract*.

**Define dimensions and sentence.** We use a tiny example so we can inspect every number.

In [ ]:
d_model, d_k = 4, 3
sentence = ["I", "like", "AI"]
print(f"d_model={d_model} (embedding size), d_k={d_k} (projection size)")
print(f"Sentence: {sentence} ({len(sentence)} tokens)")

**What do these numbers mean?** `d_model=4` means each word is represented as a 4-dimensional vector (in real Transformers this is 512 or 768). `d_k=3` is the size of Q, K, V after projection -- smaller than d_model because we are compressing into a task-specific space. We use tiny numbers here so you can trace every computation by hand.

**Create input embeddings.** In practice these come from `nn.Embedding` + positional encoding. Here we use fixed values for reproducibility.

In [ ]:
X = torch.tensor([[0.2, -0.1, 0.4, 0.1],    # "I"
                   [0.8,  0.6, 0.3, 0.5],    # "like"
                   [0.5,  0.4, 0.2, 0.7]])   # "AI"
print(f"X shape: {X.shape} -- 3 tokens, each a 4-d vector")

**Interpretation:** `X` is a $(3 \times 4)$ matrix -- 3 rows (one per token) and 4 columns (embedding dimensions). Each row is a vector that encodes *everything* the model knows about that word: its meaning, syntax, relationships -- all tangled together in 4 numbers. The challenge is that this single vector must serve three different roles (seeking, offering, carrying). That is why we need projection.

**Create three projection matrices.** Each `nn.Linear` has different random weights -- they will project X into three different spaces.

In [ ]:
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)
print(f"Each W maps {d_model}d -> {d_k}d. Three DIFFERENT matrices.")

**What are these matrices?** Each `nn.Linear(4, 3)` is a $(4 \times 3)$ weight matrix with random initial values. During training, gradient descent will adjust these weights so that:
- $W_Q$ learns to extract "what does this token **need** from others?"
- $W_K$ learns to extract "what does this token **offer** to others?"
- $W_V$ learns to extract "what **content** should be transferred?"

Right now they are random -- but the structure (three separate matrices on the same input) is already meaningful.

**Project X to get Q, K, V.** Each token now has three representations -- one per role.

In [ ]:
Q = W_Q(X)   # what does each token NEED?
K = W_K(X)   # what does each token OFFER?
V = W_V(X)   # what content does each token CARRY?
print(f"Q: {Q.shape}, K: {K.shape}, V: {V.shape}")

**What changed?** The input $X$ was $(3 \times 4)$. After projection, Q, K, V are each $(3 \times 3)$ -- the number of tokens stays the same (3), but the dimension changed from 4 to 3. This is because each $W$ selects and recombines features into a smaller, task-specific space. Think of it as: the 4d embedding knows everything, but the 3d query knows *exactly what it needs*.

> **Key question to ask yourself:** If Q, K, V all came from the same X, why do they look different? Because each $W$ is a different filter that emphasises different aspects of the same knowledge.

**Visualise the three views.** Notice each heatmap has different patterns -- each W highlights different aspects of the same knowledge.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, data, title, cmap in zip(axes,
    [X.detach(), Q.detach(), K.detach(), V.detach()],
    ['X (embedding)', 'Q (need)', 'K (offer)', 'V (carry)'],
    ['Blues', 'Purples', 'Greens', 'Oranges']):
    im = ax.imshow(data.numpy(), cmap=cmap, aspect='auto')
    ax.set_yticks(range(3)); ax.set_yticklabels(sentence)
    ax.set_xlabel('Dim'); ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax)
plt.suptitle('Same input, three different knowledge representations', y=1.02)
plt.tight_layout(); plt.show()

**How to read these heatmaps:** Each row is a token, each column is a dimension. The colour intensity shows the value in that cell.

- **X (leftmost):** all three tokens have the same type of representation -- general, untargeted
- **Q, K, V:** each has a *different* colour pattern from the same input -- proof that the three matrices extract different features

In a trained model, Q for "like" would emphasise "I need an object" while K for "AI" would emphasise "I can be an object." Here the weights are random, so the patterns are meaningless -- but the *structure* (different outputs from same input) is the key takeaway.

---
## Part 3: The Dot Product -- How Queries Find Matching Keys

$\vec{q} \cdot \vec{k} = \sum_i q_i k_i = \|q\|\|k\|\cos\theta$ measures **alignment**:

- **Aligned** ($\cos\theta \approx 1$): high score -- **relevant**
- **Orthogonal** ($\cos\theta \approx 0$): zero -- **irrelevant**
- **Opposite** ($\cos\theta \approx -1$): negative -- **anti-relevant**

**Geometric intuition.** Three 2D examples: aligned, orthogonal, and opposite vectors.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cases = [
    (np.array([1, 0.5]), np.array([0.8, 0.4]), 'Aligned'),
    (np.array([1, 0]),   np.array([0, 1]),     'Orthogonal'),
    (np.array([1, 0.5]), np.array([-0.8, -0.4]), 'Opposite')
]
for ax, (q, k, label) in zip(axes, cases):
    ax.arrow(0, 0, q[0], q[1], head_width=0.05, color='#6c5ce7', lw=2, label='q')
    ax.arrow(0, 0, k[0], k[1], head_width=0.05, color='#00b894', lw=2, label='k')
    ax.set_title(f'{label}\nq.k = {q@k:.2f}', fontsize=11)
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-0.8, 0.8); ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What to notice:**
- **Left (aligned):** both arrows point roughly the same direction → high dot product (1.12). This means the query and key are *compatible* -- the query is looking for something, and this key offers exactly that.
- **Middle (orthogonal):** arrows are at 90° → dot product is 0. They have nothing in common -- completely irrelevant to each other.
- **Right (opposite):** arrows point in opposite directions → negative dot product (-1.12). The key offers the *opposite* of what the query needs.

In self-attention, when $q_i \cdot k_j$ is large and positive, token $i$ will **pay strong attention** to token $j$. When it is near zero, token $j$ is ignored.

---
## Part 4: The Attention Formula -- Four Steps

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| Step | Operation | What it does |
|---|---|---|
| 1 | $QK^T$ | Dot products between all query-key pairs |
| 2 | $\div \sqrt{d_k}$ | Scale down to prevent softmax saturation |
| 3 | softmax | Convert to probabilities (each row sums to 1) |
| 4 | $\times V$ | Weighted sum of values -- context-enriched output |

**Step 1: Compute $QK^T$.** Entry $(i,j)$ = how much token $i$ should attend to token $j$.

In [ ]:
scores = Q @ K.T
print(f"Score matrix ({scores.shape}):")
for i, tok in enumerate(sentence):
    row = [f"{scores[i,j].item():+.3f}" for j in range(3)]
    print(f"  {tok:>4} -> [{', '.join(row)}]")

**Reading the score matrix:** Each row is a query (the token asking "who should I attend to?"). Each column is a key (the token being evaluated). The score at position $(i,j)$ is the raw dot product $q_i \cdot k_j$ -- a measure of how well token $i$'s demand aligns with token $j$'s supply.

> **Try this:** Look at the row for "like". Which token does it score highest with? That will be the token "like" pays most attention to after softmax.

**Step 2: Scale by $\sqrt{d_k}$.** Keeps scores moderate so softmax does not saturate.

In [ ]:
scores_scaled = scores / math.sqrt(d_k)
print(f"Scaling factor: sqrt({d_k}) = {math.sqrt(d_k):.2f}")
print(f"Max score: {scores.max().item():.3f} -> {scores_scaled.max().item():.3f}")

**Why does this matter?** The raw scores can be large numbers (especially when $d_k$ is big -- in real models $d_k=64$). If we feed large numbers into softmax, it produces near-one-hot outputs: one token gets 99% of the attention and all others get nearly 0%. That makes the model **overconfident** and kills gradient flow. Dividing by $\sqrt{d_k}$ shrinks the scores back to a reasonable range so softmax produces *soft* distributions where multiple tokens share the attention.

**Step 3: Softmax (row-wise).** Each row becomes a probability distribution over keys.

In [ ]:
attn_weights = F.softmax(scores_scaled, dim=-1)
for i, tok in enumerate(sentence):
    row = [f"{attn_weights[i,j].item():.3f}" for j in range(3)]
    print(f"  {tok:>4} attends to: [{', '.join(row)}]  sum={attn_weights[i].sum():.2f}")

**Reading the attention weights:** Each row is now a **probability distribution** -- it sums to 1.0. The weight $w_{i,j}$ tells us: "when computing the output for token $i$, how much should I take from token $j$'s Value vector?"

> **Notice:** The weights are fairly uniform (around 0.33 each). This is because our $d_k=3$ is tiny and the scores are close to each other. In real Transformers ($d_k=64$, trained weights), the distribution is much sharper -- some tokens get 0.6+ while others drop to near 0.

> **Self-check:** Do all rows sum to 1.0? They must, because softmax guarantees this.

**Step 4: Weighted sum of V.** Each output row = blend of all value vectors, weighted by attention.

In [ ]:
output = attn_weights @ V
print(f"Output for 'like':")
print(f"  = {attn_weights[1,0]:.3f}*V_I + {attn_weights[1,1]:.3f}*V_like + {attn_weights[1,2]:.3f}*V_AI")
print(f"  = {output[1].detach().tolist()}")

**What just happened?** The output for "like" is no longer just "like"'s own information. It is a **weighted blend** of the Value vectors of all three tokens. The attention weights determined *how much* of each token's content was mixed in.

This is the core of self-attention: **each token absorbs knowledge from every other token**, weighted by relevance. Before this step, "like" only knew about itself. After this step, "like" carries information about "I" (its subject) and "AI" (its object).

> **Key insight:** The Value vector is the **payload** -- it carries the actual semantic content. Q and K are only used for **matching** (deciding the weights). V is what actually gets blended into the output.

**Visualise the attention matrix.** Rows = who is asking, columns = who gets attended to.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(attn_weights.detach().numpy(), cmap='Blues', vmin=0, vmax=0.5)
ax.set_xticks(range(3)); ax.set_xticklabels(sentence)
ax.set_yticks(range(3)); ax.set_yticklabels(sentence)
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.set_title('Attention Weights')
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{attn_weights[i,j].item():.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

**How to read this plot:**
- **Each row** answers: "where does this token look for information?"
- **Each column** answers: "how much attention does this token receive?"
- **The numbers** are the softmax weights -- they sum to 1.0 per row
- **Darker blue** = stronger attention = more information pulled from that token

> **Exercise:** Imagine you are "like" (row 2). You need a subject and an object. Which tokens would a trained model attend to? In a real model, "like" would attend strongly to "I" (subject) and "AI" (object), with lower weight on itself.

---
## Part 5: Why Scale by $\sqrt{d_k}$?

Large $d_k$ makes dot products bigger. Softmax saturates -- one token gets all weight, gradients vanish.

**Compare softmax with and without scaling** for $d_k$ = 4, 64, 512.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, dk in zip(axes, [4, 64, 512]):
    q, k = torch.randn(dk), torch.randn(dk)
    raw = (q @ k).item()
    s = raw / math.sqrt(dk)
    sm_r = F.softmax(torch.tensor([raw, 0.5, 0.2]), dim=0).numpy()
    sm_s = F.softmax(torch.tensor([s, 0.5/math.sqrt(dk), 0.2/math.sqrt(dk)]), dim=0).numpy()
    x = np.arange(3)
    ax.bar(x-0.15, sm_r, 0.3, label='No scaling', color='#e74c3c', alpha=0.8)
    ax.bar(x+0.15, sm_s, 0.3, label='Scaled', color='#2ecc71', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(['t1','t2','t3'])
    ax.set_title(f'd_k={dk}, raw={raw:.1f}'); ax.set_ylim(0,1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What to observe:**
- At $d_k=4$: both bars look similar -- scaling barely changes anything because scores are small
- At $d_k=64$: the red bars (no scaling) start becoming uneven -- one token dominates
- At $d_k=512$: without scaling, softmax is essentially a **hard argmax** -- token 1 gets ~100%, others get ~0%. With scaling (green), the distribution stays soft and balanced

> **Why this matters for learning:** If softmax gives 1.0 to one token and 0.0 to others, the gradient of softmax is nearly zero everywhere. The model **stops learning** because it cannot adjust the weights. Scaling keeps the gradients healthy.

---
## Part 6: $QK^T \neq KQ^T$ -- Seeking is Not Offering

$(QK^T)_{i,j} = q_i \cdot k_j$ = token $i$'s **demand** vs token $j$'s **supply**.
$(KQ^T)_{i,j} = k_i \cdot q_j$ = token $i$'s **supply** vs token $j$'s **demand**.

These are different because seeking and offering are different roles.

**Numerical proof.** We compute both and show the difference.

In [ ]:
QKt = (Q @ K.T).detach().numpy()
KQt = (K @ Q.T).detach().numpy()
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
for ax, mat, t in zip([ax1,ax2,ax3], [QKt, KQt, QKt-KQt],
    ['QK^T (seek vs offer)', 'KQ^T (offer vs seek)', 'Difference']):
    im = ax.imshow(mat, cmap='RdBu_r', aspect='equal')
    ax.set_xticks(range(3)); ax.set_xticklabels(sentence)
    ax.set_yticks(range(3)); ax.set_yticklabels(sentence); ax.set_title(t)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print("Non-zero difference confirms: seeking and offering are different operations.")

**Why is this important?** If we used the same matrix for Q and K (i.e., $W_Q = W_K$), then $QK^T$ would be **symmetric** -- token A attending to token B would equal token B attending to token A. But language is not symmetric: "the cat chased the dog" has a *chaser* and a *chasee*. By using different $W_Q$ and $W_K$, the model can learn **complementary** roles: one token can strongly seek what another token offers, without the reverse being true.

> **Analogy:** In a job market, an employer's "what I need" (query) is different from a candidate's "what I offer" (key). Matching is inherently asymmetric -- that is why we need two separate matrices.

---
## Part 7: Self-Attention as Knowledge Discovery

**Before:** each token has isolated knowledge.
**After:** each token is enriched with context from the entire sentence.

The pipeline: $W_Q$ expresses demand, $W_K$ expresses supply, $Q \cdot K$ measures match quality, $V$ is the payload transferred.

**Set up a longer sentence** to see more meaningful attention patterns.

In [ ]:
torch.manual_seed(7)
sentence2 = ["What", "is", "the", "capital", "of", "Saudi", "Arabia"]
N = len(sentence2)
print(f"Sentence: {sentence2} ({N} tokens)")

**Create embeddings and projections.**

In [ ]:
d2, dk2 = 16, 8
X2 = torch.randn(N, d2)
W_Q2 = nn.Linear(d2, dk2, bias=False)
W_K2 = nn.Linear(d2, dk2, bias=False)
W_V2 = nn.Linear(d2, dk2, bias=False)

**Run the attention formula.**

In [ ]:
Q2, K2, V2 = W_Q2(X2), W_K2(X2), W_V2(X2)
scores2 = Q2 @ K2.T / math.sqrt(dk2)
attn2 = F.softmax(scores2, dim=-1)
output2 = attn2 @ V2

**Visualise attention.** In a trained model, *capital* would attend strongly to *Saudi* and *Arabia*.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(attn2.detach().numpy(), cmap='Blues', vmin=0)
ax.set_xticks(range(N)); ax.set_xticklabels(sentence2, rotation=45, ha='right')
ax.set_yticks(range(N)); ax.set_yticklabels(sentence2)
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.set_title('Attention: "What is the capital of Saudi Arabia"')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{attn2[i,j].item():.2f}', ha='center', va='center', fontsize=9,
                color='white' if attn2[i,j].item() > 0.25 else 'black')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

**Interpretation:** Since the weights are random (untrained), the attention patterns do not reflect real language understanding. But the **structure** is already visible:
- Each row is a probability distribution (sums to 1.0)
- Some tokens get more attention than others, even randomly
- The $7 \times 7$ matrix shows that self-attention scales to any sequence length

**In a trained model**, you would see:
- "capital" attending strongly to "Saudi" and "Arabia" (learning which country)
- "Saudi" and "Arabia" attending to each other (the model binds them as one entity)
- "the" attending to "capital" (article finds its noun)

Self-attention discovers **semantic relationships** without any explicit grammar rules -- it learns them from data.

> **Exercise:** If you ran this on "The cat sat on the mat", which token pairs would you expect a trained model to link?

---
## Part 8: Complete Single-Head Self-Attention

All four steps in one clean function.

In [ ]:
def self_attention(X, W_Q, W_K, W_V, mask=None):
    Q, K, V = W_Q(X), W_K(X), W_V(X)
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

**Understanding the function line by line:**
1. `Q, K, V = W_Q(X), W_K(X), W_V(X)` -- project X into three spaces (the three roles)
2. `scores = Q @ K.T / math.sqrt(d_k)` -- compute all pairwise scores and scale them
3. `if mask: scores = scores + mask` -- optionally block future positions (for GPT-style models)
4. `weights = F.softmax(scores, dim=-1)` -- convert to probabilities per row
5. `return weights @ V` -- weighted blend of value vectors = context-enriched output

This is the **entire** self-attention mechanism in 5 lines. Every modern LLM (GPT-4, LLaMA, Gemini) uses exactly this formula at its core.

**Verify** it matches our step-by-step results.

In [ ]:
out, w = self_attention(X, W_Q, W_K, W_V)
print(f"Input: {X.shape} -> Output: {out.shape}")
print(f"Matches step-by-step? {torch.allclose(out, output, atol=1e-6)}")

---
## Part 9: Causal Masking -- Blocking the Future

In autoregressive models (GPT), each token must **not see the future**. The causal mask adds $-\infty$ to future positions. Since $\text{softmax}(-\infty) = 0$, masked positions get zero weight.

**Create the causal mask.** Upper triangle = blocked.

In [ ]:
sentence3 = ["The", "cat", "is", "sleeping"]
N3 = len(sentence3)
causal_mask = torch.triu(torch.ones(N3, N3) * float('-inf'), diagonal=1)
for i, t in enumerate(sentence3):
    row = ['  0  ' if causal_mask[i,j]==0 else '-inf ' for j in range(N3)]
    print(f"  {t:>8} [{' '.join(row)}]")

**Reading the mask pattern:** The diagonal and below (lower triangle) are 0 -- these positions are **allowed**. Above the diagonal (upper triangle) is $-\infty$ -- these are **blocked**.

- Row 0 ("The"): only position 0 is allowed → can only see itself
- Row 1 ("cat"): positions 0-1 allowed → sees "The" and itself
- Row 2 ("is"): positions 0-2 allowed → sees "The", "cat", itself
- Row 3 ("sleeping"): all positions allowed → sees everything

This enforces the autoregressive constraint: **each token can only attend to tokens that came before it** (and itself). When predicting "sleeping", the model knows about "The cat is" but not future tokens.

**Run attention with and without the mask.**

In [ ]:
torch.manual_seed(42)
X3 = torch.randn(N3, 8)
W_Q3 = nn.Linear(8, 4, bias=False)
W_K3 = nn.Linear(8, 4, bias=False)
W_V3 = nn.Linear(8, 4, bias=False)

_, w_full = self_attention(X3, W_Q3, W_K3, W_V3)
_, w_causal = self_attention(X3, W_Q3, W_K3, W_V3, mask=causal_mask)

**Compare.** The masked version has zeros above the diagonal -- each token sees only the past.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
for ax, w, title in zip([ax1,ax2], [w_full.detach().numpy(), w_causal.detach().numpy()],
    ['Without Mask', 'With Causal Mask']):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=0.6)
    ax.set_xticks(range(N3)); ax.set_xticklabels(sentence3, rotation=45, ha='right')
    ax.set_yticks(range(N3)); ax.set_yticklabels(sentence3); ax.set_title(title)
    for i in range(N3):
        for j in range(N3):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if w[i,j]>0.3 else 'black')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Comparing the two plots:**
- **Left (no mask):** every token can see every other token. Weights are spread across all 4 positions. This is what **encoder** models (BERT) use -- full bidirectional attention.
- **Right (causal mask):** the upper triangle is zero. "The" puts all its weight on itself (1.00) because it has no past context. "sleeping" distributes weight across all 4 tokens because it can see the full history.

> **Key takeaway:** The mask does not change the attention mechanism itself -- it just forces certain scores to $-\infty$ before softmax, which makes those positions get exactly zero weight. The formula is identical; only the input to softmax changes.

> **When is each used?**
> - **No mask:** BERT (encoder), for understanding tasks (classification, NER)
> - **Causal mask:** GPT (decoder), for generation tasks (text completion, chatbots)

---

## Part 10: The $O(N^2)$ Complexity -- Why Long Sequences Are Expensive

### Where does the cost come from?

The matrix multiplication $QK^T$ computes the dot product between **every pair** of tokens:

- $Q$ has shape $(N \times d_k)$ -- one row per token
- $K^T$ has shape $(d_k \times N)$ -- one column per token
- $QK^T$ has shape $(N \times N)$ -- one score per **pair**

That means the number of scores grows as $N^2$. If you double the sequence length, you **quadruple** the number of scores, the memory, and the compute.

### Why is this a problem?

The model parameters ($W_Q, W_K, W_V$, FFN weights, etc.) are **constant** -- they do not grow with sequence length. But the attention matrix grows **quadratically**. For long documents, the attention matrix can become larger than the model itself.

**Let us compute the numbers.** We calculate the attention matrix size and memory usage for realistic sequence lengths, and compare it to model parameter memory.

In [ ]:
seq_lengths = [512, 2048, 8192, 32768, 131072]
print(f"{'N':>8}  {'Scores':>12}  {'Memory (MB)':>12}  {'vs 512':>8}")
print("-" * 48)
for n in seq_lengths:
    scores = n * n
    memory_mb = scores * 4 / (1024**2)   # float32 = 4 bytes
    ratio = (n / 512) ** 2
    print(f"{n:>8,}  {scores:>12,}  {memory_mb:>10.1f} MB  {ratio:>7.0f}x")

**Reading the table:** Going from 512 to 2048 tokens (4x longer) costs 16x more memory. Going from 512 to 32K tokens (64x longer) costs 4,096x more. At 128K tokens (the context window of GPT-4 and Claude), a **single** attention matrix in **one** layer takes ~64 GB in float32 -- that is the entire memory of an A100 GPU, just for one matrix.

> **This is why efficient attention is a major research area.** Techniques like Flash Attention, sparse attention, and linear attention all aim to reduce this quadratic cost.

> **Important nuance:** This is the cost for one attention head in one layer. Real models have 12-96 layers with 12-128 heads each. The total cost is multiplied accordingly.

**Visualising quadratic growth.** We plot the attention matrix size as sequence length increases. The linear line shows how model parameters grow (they do not -- they are fixed). The quadratic curve shows how the attention cost explodes.

In [ ]:
ns = np.array([128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768])
attn_memory_mb = (ns ** 2) * 4 / (1024**2)   # float32
param_memory_mb = np.full_like(attn_memory_mb, 1400)  # ~1.4 GB for a 350M param model

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ns, attn_memory_mb, 'o-', color='#e74c3c', lw=2, ms=6, label='Attention matrix (one head, one layer)')
ax.axhline(1400, color='#2ecc71', lw=2, ls='--', label='Model parameters (~350M, fixed)')
ax.axhline(80000, color='gray', lw=1, ls=':', label='A100 GPU memory (80 GB)')
ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('Quadratic Attention Cost vs Fixed Model Size', fontsize=13)
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

**How to read this plot:**
- The **red curve** (attention matrix) rises steeply -- it is quadratic on the log-log scale
- The **green dashed line** (model parameters) is flat -- model size does not depend on sequence length
- The **gray dotted line** is the total memory of an A100 GPU (80 GB)

At short sequences (N=512), the attention matrix is tiny compared to the model. But at long sequences (N=32K+), the attention matrix **dominates** and eventually exceeds GPU memory. This is the fundamental bottleneck of Transformers.

> **Self-check question:** If you double the sequence length from 4K to 8K, by how much does the attention memory increase? (Answer: $2^2 = 4$ times)

**Measuring it directly.** Let us time how long `QK^T` takes for increasing sequence lengths on your machine. You will see the quadratic growth in real wall-clock time.

In [ ]:
import time

test_lengths = [256, 512, 1024, 2048, 4096]
d_test = 64
times_ms = []

for n in test_lengths:
    q_test = torch.randn(n, d_test)
    k_test = torch.randn(n, d_test)
    # Warm up
    _ = q_test @ k_test.T
    # Measure
    t0 = time.perf_counter()
    for _ in range(10):
        _ = q_test @ k_test.T
    elapsed = (time.perf_counter() - t0) / 10 * 1000
    times_ms.append(elapsed)
    print(f"N={n:>5}: QK^T takes {elapsed:.2f} ms  (matrix size: {n}x{n} = {n*n:,} scores)")

**Plot the timing results.** We overlay the measured times with a theoretical $O(N^2)$ curve to confirm the quadratic scaling.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(test_lengths, times_ms, 'o-', color='#e74c3c', lw=2, ms=8, label='Measured time')

# Overlay theoretical O(N^2) curve, scaled to match first point
scale = times_ms[0] / (test_lengths[0] ** 2)
theoretical = [scale * n**2 for n in test_lengths]
ax.plot(test_lengths, theoretical, '--', color='gray', lw=1.5, label='Theoretical O(N$^2$)')

ax.set_xlabel('Sequence Length (N)', fontsize=12)
ax.set_ylabel('Time per QK^T (ms)', fontsize=12)
ax.set_title('Self-Attention Time Grows Quadratically', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)

# Annotate the doubling ratios
for i in range(1, len(test_lengths)):
    ratio = times_ms[i] / times_ms[i-1]
    ax.annotate(f'{ratio:.1f}x', xy=(test_lengths[i], times_ms[i]),
                xytext=(0, 15), textcoords='offset points',
                fontsize=9, color='#e74c3c', ha='center')
plt.tight_layout(); plt.show()

**Interpretation of the timing plot:**
- The red dots follow the gray dashed $O(N^2)$ curve closely -- confirming quadratic scaling
- The annotations show the **ratio** when doubling N: each doubling should give roughly **4x** the time
- At N=4096, the computation is already noticeably slow -- and real models use N=8K to 128K

> **Why this matters in practice:** A model like GPT-4 with 128K context window, 96 layers, and 96 heads would need to compute $96 \times 96 = 9{,}216$ attention matrices of size $128K \times 128K$ for a single forward pass. Without optimisations like Flash Attention and KV-caching, this would be impossibly slow.

### What solutions exist?

| Approach | Idea | Tradeoff |
|---|---|---|
| **Flash Attention** | Compute attention in tiles, never materialise the full $N \times N$ matrix | Exact same result, just faster and less memory |
| **Sparse Attention** | Only compute scores for nearby tokens + a few global ones | Approximate -- misses some long-range connections |
| **Linear Attention** | Replace softmax with a kernel trick to get $O(N)$ cost | Changes the attention mechanism -- different behaviour |
| **KV-Cache** (inference) | Cache K and V from previous tokens during generation | Only helps at inference time, not training |
| **Sliding Window** | Each token only attends to the last $W$ tokens | Misses connections beyond the window |

> **Key takeaway:** The quadratic cost is the **fundamental bottleneck** of Transformers. Every major advance in long-context models (GPT-4 128K, Claude 200K, Gemini 1M) required engineering solutions to manage this cost. The attention formula itself has not changed since 2017 -- what changed is how efficiently we compute it.

---

## Part 11: Two Modes of Self-Attention -- Representation vs Generation

The same self-attention formula powers two fundamentally different tasks. The only difference is the **mask**. But that small difference creates two entirely different ways of understanding language.

### Mode 1: Full Attention (Encoder / Representation)

**Used in:** BERT, RoBERTa, DeBERTa -- models that **understand** text

**The idea:** Every token can see **every other token**, including tokens that come after it. This is called **bidirectional** attention.

**Why it matters for understanding:** Consider the sentence:

> "The bank by the **river** was flooded"

To understand what "bank" means, you need to see "river" -- which comes *after* "bank." If you blocked the future, "bank" could only see "The" and would have no idea whether it means a financial institution or a riverbank. Full attention lets "bank" look at the entire sentence and resolve the ambiguity.

**Intuition:** Think of reading a paragraph for comprehension. You read it **all**, then go back and forth to connect ideas. You are not generating word by word -- you are building a complete mental picture. That is what encoder models do: they produce a **rich representation** of the entire input at once.

**Typical tasks:**
- **Classification:** "Is this email spam?" → read the whole email, then decide
- **Named Entity Recognition:** "Is *bank* a location or an organisation?" → need full context
- **Semantic similarity:** "Do these two sentences mean the same thing?" → compare full representations

**Let us see this.** We run full (bidirectional) attention on "The bank by the river was flooded" and show how "bank" attends to every other token -- including "river" which comes after it.

In [ ]:
torch.manual_seed(10)
sent_enc = ["The", "bank", "by", "the", "river", "was", "flooded"]
N_enc = len(sent_enc)

X_enc = torch.randn(N_enc, 16)
W_Qe = nn.Linear(16, 8, bias=False)
W_Ke = nn.Linear(16, 8, bias=False)
W_Ve = nn.Linear(16, 8, bias=False)

out_enc, w_enc = self_attention(X_enc, W_Qe, W_Ke, W_Ve, mask=None)

**Visualise the full attention matrix.** Focus on row 1 ("bank") -- it can see ALL tokens including "river" (position 4) which comes after it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(w_enc.detach().numpy(), cmap='Blues', vmin=0)
ax.set_xticks(range(N_enc)); ax.set_xticklabels(sent_enc, rotation=45, ha='right')
ax.set_yticks(range(N_enc)); ax.set_yticklabels(sent_enc)
ax.set_xlabel('Key (attended to)'); ax.set_ylabel('Query (asking)')
ax.set_title('Encoder (Full Attention) -- BERT style\n"bank" can see "river" (future token)')
for i in range(N_enc):
    for j in range(N_enc):
        v = w_enc[i,j].item()
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8,
                color='white' if v > 0.2 else 'black')
# Highlight "bank" row
ax.axhline(0.5, color='orange', lw=2, ls='--'); ax.axhline(1.5, color='orange', lw=2, ls='--')
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

**What to notice (orange highlighted row = "bank"):** "bank" distributes attention across ALL 7 tokens -- including "river" and "flooded" which come after it. In a trained BERT model, "bank" would attend heavily to "river" to disambiguate its meaning. Without full attention, this disambiguation would be impossible.

> **The key benefit of full attention for representation:** every token builds its representation using the **entire** sentence as context. The output representation of "bank" is not just "bank" -- it is "bank-in-the-context-of-a-river-flooding." This rich, contextualised representation is what makes BERT so powerful for classification, search, and understanding tasks.

---

### Mode 2: Causal Attention (Decoder / Autoregressive Generation)

**Used in:** GPT, LLaMA, Mistral, Claude -- models that **generate** text

**The idea:** Each token can only see tokens that came **before** it (and itself). Future tokens are invisible. This is called **causal** or **autoregressive** attention.

**Why it matters for generation:** When generating text, the model produces one token at a time. When it is deciding what comes after "The cat is", it must **not** peek at "sleeping" because "sleeping" has not been generated yet. If it could see the answer, it would just memorise instead of learning to predict.

**Intuition:** Think of writing an essay. When you write word 5, you can look back at words 1-4 for coherence, but you cannot look ahead because those words do not exist yet. You must *predict* the next word based only on what you have written so far. That is autoregressive generation.

**The training trick:** During training, the model sees the entire sentence at once (for efficiency), but the causal mask **simulates** generating one token at a time by blocking future positions. This way, we get the parallelism of self-attention with the autoregressive constraint of generation.

**Typical tasks:**
- **Text generation:** "The weather today is..." → predict "sunny"
- **Chatbots:** read the conversation so far → generate the next response
- **Code completion:** see the code written so far → predict the next line

**Let us see causal attention on the same sentence.** Now "bank" can only attend to "The" and itself. It **cannot** see "river" -- so it has no way to know it is a riverbank, not a financial bank.

In [ ]:
causal_mask_enc = torch.triu(torch.ones(N_enc, N_enc) * float('-inf'), diagonal=1)
_, w_causal_enc = self_attention(X_enc, W_Qe, W_Ke, W_Ve, mask=causal_mask_enc)

**Side by side: Encoder vs Decoder on the same sentence.** Left = full attention (BERT). Right = causal attention (GPT). Same weights, same input -- only the mask differs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

for ax, w, title in zip([ax1, ax2],
    [w_enc.detach().numpy(), w_causal_enc.detach().numpy()],
    ['ENCODER (Full Attention)\nEvery token sees everything\nUsed for: understanding, classification',
     'DECODER (Causal Attention)\nEach token sees only the past\nUsed for: generation, chatbots']):
    im = ax.imshow(w, cmap='Blues', vmin=0)
    ax.set_xticks(range(N_enc)); ax.set_xticklabels(sent_enc, rotation=45, ha='right')
    ax.set_yticks(range(N_enc)); ax.set_yticklabels(sent_enc)
    ax.set_xlabel('Key'); ax.set_ylabel('Query'); ax.set_title(title, fontsize=10)
    for i in range(N_enc):
        for j in range(N_enc):
            v = w[i,j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if v > 0.2 else 'black')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

**Compare the two plots carefully:**

| | Encoder (Left) | Decoder (Right) |
|---|---|---|
| **"bank" (row 1)** | Sees all 7 tokens -- can use "river" to disambiguate | Sees only "The" and itself -- no way to know it is a riverbank |
| **"flooded" (last row)** | Sees everything -- full context | Sees all 6 prior tokens -- nearly full context (only misses nothing here since it is last) |
| **"The" (row 0)** | Sees everything | Sees only itself -- no context at all |
| **Shape** | Full matrix (all cells non-zero) | Lower triangle only (upper triangle is zero) |

> **The fundamental trade-off:**
> - **Encoder** gives richer representations (every token sees everything) but **cannot generate** text -- it has no notion of "next token"
> - **Decoder** can generate text token by token but each token has **less context** (can only look backwards)
>
> This is why modern models like GPT-4 and Claude use decoders: they sacrifice some representation quality for the ability to generate. The causal constraint is the price of generation.

### When Would You Choose Each?

| Task | Best mode | Why |
|---|---|---|
| "Is this review positive or negative?" | **Encoder** | Need to read the whole review before deciding |
| "Translate this sentence" | **Encoder-Decoder** | Encoder reads the input fully, decoder generates the output |
| "Complete this sentence: The cat..." | **Decoder** | Must generate one word at a time, left to right |
| "Summarise this article" | **Decoder** | Generates the summary word by word |
| "Find all person names in this text" | **Encoder** | Need full context to classify each token |
| "Chat with the user" | **Decoder** | Generates each response token autoregressively |

> **Key intuition:** If the task requires **generating** new text, use a decoder (causal mask). If the task requires **understanding** existing text, use an encoder (full attention). If the task requires both (e.g., translation), use encoder-decoder.

---
## Summary

| Concept | Key Point |
|---|---|
| **Q** | "What do I need?" -- demand |
| **K** | "What can I offer?" -- supply |
| **V** | "What do I carry?" -- payload |
| **$QK^T$** | Asymmetric: $QK^T \neq KQ^T$ |
| **$/\sqrt{d_k}$** | Prevents softmax saturation |
| **Softmax** | Scores -> probabilities |
| **Output** | Weighted blend of V |
| **Causal mask** | Blocks future tokens |

$$X \xrightarrow{W_Q,W_K,W_V} Q,K,V \xrightarrow{QK^T/\sqrt{d_k}} \text{scores} \xrightarrow{\text{softmax}} \text{weights} \xrightarrow{\times V} \text{output}$$

Each token enters knowing only itself. It leaves knowing the entire sentence.